In [64]:
import pandas as pd
import matplotlib.pyplot as plt

## Uhm, what? 

In [65]:
def load_data():
    df1 = pd.read_csv("../data/df_test.csv") 
    df2 = pd.read_csv("../data/df_train.csv")
    df = pd.concat([df1, df2], ignore_index=True)
    return df

df = load_data()

In [66]:
df.head()

,review_text,clean_text,review_date,clean_length,tokenized,sentiment_label
0,barang bagus proses cepat,barang bagus proses cepat,2025-07-11,4,"['barang', 'bagus', 'proses', 'cepat']",positive
1,Bagus sesuai,bagus sesuai,2025-03-07,2,"['bagus', 'sesuai']",positive
2,Cepat kirimnya dalam sejam. Porsi medium cukup...,cepat kirimnya dalam sejam porsi medium cukup ...,2021-09-04,11,"['cepat', 'kirimnya', 'dalam', 'sejam', 'porsi...",positive
3,"kaos kakinya nyaman dipake, rubber di bagian b...",kaos kakinya nyaman dipakai rubber di bagian b...,2024-07-15,12,"['kaos', 'kakinya', 'nyaman', 'dipakai', 'rubb...",positive
4,Pengiriman lama sekali padahal jarak begitu dekat,pengiriman lama sekali padahal jarak begitu dekat,2022-09-07,7,"['pengiriman', 'lama', 'sekali', 'padahal', 'j...",neutral


## Complaint Topic Modeling
1. Nah, just use LDA for now (kapan2 use BERTopic)
2. Using vectorizer, with max_df 0.95, and min_df = 5, with indonesian stopwords, added with custom stopwords and domain stopwords, 
an ngram range 1, 2
3. Vectorizer fit transform to all data, 
4. but then transform again for negative text sentiment --> do LDA for negative 
5. and also transform again for positive text --> do LDA for positive
6. So 1 vectorizer and 2 lda model. 
7. 3 topics: product quality, delivery & packaging, and product matching


In [67]:
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# it's only have 798 negative review. shall we just do LDA and BERTopic? Let's see later. 
df_neg = df[df.sentiment_label=="negative"]
df_pos = df[df.sentiment_label=="positive"]

In [68]:
# stopwords
from nltk.corpus import stopwords
list_stopwords = stopwords.words('indonesian')

In [69]:
custom_stopwords = [
    "nya", "aja", "sih", "dong", "lah", "kah", "deh",
    "yg", "ga", "gak", "nggak", "kalo", "kalau",
    "udah", "sudah", "bgt", "banget", "ya"
]

domain_stopwords = [
    "barang", "produk", "beli", "seller", "penjual",
    "pembeli", "tokopedia", "item", "pakai", "pas", "kali", "kasih"
]

all_stopwords = list(set(list_stopwords + custom_stopwords + domain_stopwords))

In [ ]:
vectorizer = CountVectorizer(
    max_df=0.95,      # ignore very common words
    min_df=5,         # ignore very rare words
    stop_words=all_stopwords,   # or use Indonesian stopwords if you have
    ngram_range = (1,2)
)


### It's Negative

In [71]:
vectorizer.fit_transform(df.clean_text)
x_neg = vectorizer.transform(df_neg.clean_text)
x_pos = vectorizer.transform(df_pos.clean_text)

lda_neg = LatentDirichletAllocation(
    n_components=3,   # number of topics
    random_state=42
)
lda_pos = LatentDirichletAllocation(
    n_components=3,   # number of topics
    random_state=42
)

lda_neg.fit(x_neg)
lda_pos.fit(x_pos)

c:\Users\ACER\OneDrive\Jobless\Projects\customer-review\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0
,"perp_tol perp_tol: float, default=1e-1Perplexity tolerance. Only used when ``evaluate_every`` is greater than 0.",0.1


In [72]:
# oh it's a fast one. lol.
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx + 1}:")
        print(" ".join([
            feature_names[i]
            for i in topic.argsort()[:-no_top_words - 1:-1]
        ]))

In [73]:
feature_names = vectorizer.get_feature_names_out()

display_topics(lda_neg, feature_names, 15)
topic_results = lda_neg.transform(x_neg)

df_neg["dominant_topic"] = topic_results.argmax(axis=1)
df_neg["dominant_topic"] = df_neg["dominant_topic"].map({0: "Product Quality Issues", 1: "Delivery Issues", 2: "Product Mismatch"})


Topic 1:
dikirim pecah bahan pengiriman pesan rusak bagus telur jelek kemasan busuk ukuran toko amp bocor

Topic 2:
kirim sesuai dikirim chat pesan kurir kecewa pesanan dipakai rusak warna harga komplain bintang barangnya

Topic 3:
pengiriman sesuai daging kecewa kayak super kualitas pesan lemak bagus harga warna bau toko respon


### Conclusion on Negative
1. Product quality issues (damaged, spoiled, poor quality)
2. Delivery & logistics issues (shipping delays) 
3. Under expectations (wrong size, color, or material)

### It's for Positive

In [74]:
feature_names = vectorizer.get_feature_names_out()
display_topics(lda_pos, feature_names, 10)

topic_results = lda_pos.transform(x_pos)

df_pos["dominant_topic"] = topic_results.argmax(axis=1)
df_pos["dominant_topic"] = df_pos["dominant_topic"].map({0: "Accurate Orders", 1: "Good Product Quality", 2: "Fast Delivery & Packaging"})


Topic 1:
bagus kualitas sesuai enak bahan ukuran oke suka nyaman kualitas bagus

Topic 2:
mantap ok fast terima good respon puas order toko langsung

Topic 3:
cepat sesuai pengiriman pengiriman cepat aman packing terima pesanan deskripsi rapi


### Conclusion for Positive
1. Products as expected 
2. Good product quality
3. Fast delivery

In [75]:
df["dominant_topic"] = None

# put the neg_df dominant topic back to df
df.loc[df_neg.index, "dominant_topic"] = df_neg["dominant_topic"]
df.loc[df_pos.index, "dominant_topic"] = df_pos["dominant_topic"]

In [76]:
# save the df into final_df
df.to_csv("../data/df_final.csv", index=False)

In [77]:
# save vectorizer, and lda models
import joblib
joblib.dump(vectorizer, "../models/vectorizer.joblib")
joblib.dump(lda_neg, "../models/lda_neg.joblib")
joblib.dump(lda_pos, "../models/lda_pos.joblib")

['../models/lda_pos.joblib']

### And by chatgpt
“Topic modeling reveals that customer dissatisfaction is primarily driven by product quality issues (damaged, spoiled, or poor quality products), delivery problems, and mismatches between expectations and received items."

In [ ]:
# how about using vectorizer for df_all
# and then lda1 for negative
# and model lda2 for positive. 
# so whether we use negative or positive. it differs. right? 